# 🧬 ProGen2 Tools Example

This notebook demonstrates how to generate and score protein sequences using **ProGen2**.

## 📖 What is ProGen2?

[ProGen2](https://www.cell.com/cell-systems/fulltext/S2405-4712(23)00272-7) is a large autoregressive protein language model trained on millions of protein sequences for de novo protein generation.

### Key Features:
- 🧪 **Generation** - Autoregressive protein sequence generation from prompts
- 📏 **Scoring** - Autoregressive likelihood scoring to assess sequence naturalness
- ⚡ **GPU-ready** - Optimized for CUDA execution with efficient batching
- ⚙️ **Flexible models** - Multiple model sizes available, from 151M to 6B parameters

## 📦 Imports


## Input/Output Schema

### `ProGen2SampleInput`
| Field | Type | Description |
|-------|------|-------------|
| prompts | List[str] | Prompt sequences for protein generation |

### `ProGen2SampleConfig`
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| model_checkpoint | str | "progen2-large" | ProGen2 model checkpoint to use |
| temperature | float | 0.2 | Scales the randomness of sampling |
| top_p | float | 0.95 | Nucleus sampling parameter |
| top_k | int | 0 | Top-k sampling limit (0 to disable) |
| max_length | int | 256 | Maximum total sequence length including prompt |
| truncate_at_stop | bool | True | Whether to truncate sequences at stop tokens |
| strip_special_tokens | bool | True | Whether to strip start and stop tokens ('1' or '2') |
| prepend_prompt | bool | True | Whether to prepend prompt to generation |
| batch_size | Optional[int] | None | Max number of samples on the GPU at once |
| return_logits | bool | False | Whether to include per-position logits in the output |

### `ProGen2SampleOutput`
| Field | Type | Description |
|-------|------|-------------|
| sequences | List[str] | Generated protein sequences |
| logits | Optional[List[List[List[float]]]] | Per-position logits for generated tokens |

### `ProGen2ScoringInput`
| Field | Type | Description |
|-------|------|-------------|
| sequences | List[str] | Protein sequences to score |

### `ProGen2ScoringConfig`
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| model_checkpoint | str | "progen2-large" | ProGen2 model checkpoint to use |
| batch_size | Optional[int] | None | Max number of samples on the GPU at once |
| return_logits | bool | False | Whether to include per-position logits in the output |

### `ProGen2ScoringOutput` (alias: `CausalModelScoringOutput`)
| Field | Type | Description |
|-------|------|-------------|
| scores | List[SequenceScores] | List of scoring outputs, one per input sequence |

### `SequenceScores`
| Field | Type | Description |
|-------|------|-------------|
| metrics | Dict[str, float] | Dictionary of scalar scoring metrics (log_likelihood, avg_log_likelihood, perplexity) |
| logits | Optional[List[List[float]]] | Per-position logits array (seq_len, vocab_size) |
| vocab | Optional[List[str]] | Token ordering for logits |

In [1]:
from bio_programming.bio_tools.tools.causal_models.progen2 import (
    run_progen2_sample, ProGen2SampleInput, ProGen2SampleConfig,
    run_progen2_score, ProGen2ScoringInput, ProGen2ScoringConfig,
)


## 🧪 1. Generate Protein Sequences

Generate protein sequences from a prompt using autoregressive sampling.

ProGen2 extends a given protein prompt by predicting the most likely amino acids one at a time, using its learned distribution over millions of natural protein sequences. The sampling parameters control the diversity and quality of the generated output.

### Configuration options:
- 🧠 **`model_checkpoint`** - ProGen2 model variant (e.g., `progen2-large` for the 2B parameter model)
- 🧬 **`max_length`** - Maximum total sequence length including prompt
- 🌡️ **`temperature`** - Controls diversity of generation (lower = more conservative, higher = more diverse)
  - `0.2` - Conservative, high-confidence predictions (recommended for proteins)
  - `0.5` - Moderate diversity
  - `1.0` - High diversity, more exploratory generation
- 🎯 **`top_p`** - Nucleus sampling threshold (cumulative probability cutoff)
- 🎯 **`top_k`** - Restrict sampling to the top-k most likely tokens
- ✂️ **`truncate_at_stop`** - Whether to truncate at terminal tokens
- 🧹 **`strip_special_tokens`** - Whether to remove start/stop tokens from output


In [2]:
inputs = ProGen2SampleInput(prompts=["MKTAYIAKQRQISF"])

# Configure generation parameters
config = ProGen2SampleConfig(
    model_checkpoint="progen2-large",
    max_length=120,
    temperature=0.2,
    top_p=0.95,
    top_k=0,
    truncate_at_stop=True,
    strip_special_tokens=True,
)

sample_result = run_progen2_sample(inputs, config)

# Display the generated sequence
prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} residues")


Prompt:       1MKTAYIAKQRQISF
Generated:    MKTAYIAKQRQISFLLLLLLLLPLLSACGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGG...
Total length: 119 residues


## 📏 2. Score Sequences

Compute autoregressive likelihood metrics for protein sequences.

ProGen2 scores sequences by computing the autoregressive log-likelihood at each position. The resulting perplexity measures how "natural" a protein sequence looks to the model. Lower values indicate sequences that are more consistent with the patterns learned from natural proteins. This is useful for ranking designed sequences or evaluating sequence quality.

### Configuration options:
- 🧠 **`model_checkpoint`** - ProGen2 model variant (e.g., `progen2-large` for the 2B parameter model)
- 📦 **`batch_size`** - Number of sequences to score in parallel
- 📊 **`return_logits`** - Whether to return per-position amino acid log-probabilities


In [3]:
score_inputs = ProGen2ScoringInput(sequences=["MVLSPADKTN", "MKTLLILAVVAA"])

# Configure scoring parameters
score_config = ProGen2ScoringConfig(batch_size=2, return_logits=False)

score_result = run_progen2_score(score_inputs, score_config)
print(f"Sequence 1: {score_inputs.sequences[0]}")
print(f"    Perplexity: {score_result.scores[0].metrics['perplexity']:.3f}")
print(f"Sequence 2: {score_inputs.sequences[1]}")
print(f"    Perplexity: {score_result.scores[1].metrics['perplexity']:.3f}")


Sequence 1: MVLSPADKTN
    Perplexity: 14.346
Sequence 2: MKTLLILAVVAA
    Perplexity: 7.641


## 💾 3. Export Results

Save the results to files for downstream analysis.

### Supported formats:
- 📄 **JSON** - Structured data with metadata and all scoring information
- 🧬 **FASTA** - Simple sequence format for compatibility with other bioinformatics tools

The exported results can be used for:
- Downstream sequence analysis and annotation (generated sequences)
- Ranking and filtering candidate sequences (scoring metrics)
- Input to other bioinformatics pipelines

In [ ]:
# Export generated sequences to FASTA format
sample_result.export("./example_output/progen2_sequences", file_format="fasta")
print("Generated sequences exported to ./example_output/progen2_sequences")

# Export scores to JSON format
score_result.export("./example_output/progen2_scores", file_format="json")
print("Scores exported to ./example_output/progen2_scores")